In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc file dữ liệu sạch từ bước trước
df = pd.read_csv('data_cleaned/dataset_moodbite_clean.csv')

# --- LƯU Ý KỸ ---
# Tool Apify khi xuất CSV có thể chia review thành nhiều cột (VD: reviews/0/text, reviews/1/text...)
# Hoặc gộp chung vào 1 cột. Ở đây ta lấy TẤT CẢ chữ trong mỗi dòng ghép lại thành 1 đoạn văn bản lớn.
df['all_text'] = df.apply(lambda row: ' '.join(row.values.astype(str)).lower(), axis=1)

# 2. Xây dựng Bộ từ điển Cảm xúc (Mood Lexicon)
# Đây là minh chứng học thuật để bạn bảo vệ việc: "Em phân cụm dựa trên cái gì?"
mood_dictionaries = {
    'comfort_cozy': ['chill', 'thư giãn', 'yên tĩnh', 'thoải mái', 'ấm cúng', 'tâm tình', 'nhẹ nhàng', 'view đẹp'],
    'spicy_hot': ['cay', 'nóng', 'tê', 'đậm đà', 'xuýt xoa', 'sa tế', 'ớt'],
    'fresh_healthy': ['tươi', 'thanh mát', 'sạch', 'healthy', 'rau', 'healthy', 'ngọt tự nhiên'],
    'cheap_budget': ['rẻ', 'bình dân', 'sinh viên', 'hợp lý', 'phải chăng', 'vỉa hè'],
    'quick_fast': ['nhanh', 'vội', 'tiện', 'lấy luôn', 'không phải đợi', 'ăn liền']
}

# 3. Hàm tính điểm Cảm xúc (Scoring Function)
def calculate_mood_score(text, keywords):
    score = 0
    for word in keywords:
        # Đếm số lần từ khóa xuất hiện trong toàn bộ text của quán đó
        score += text.count(word)
    return score

# 4. Áp dụng chấm điểm cho từng quán
for mood, keywords in mood_dictionaries.items():
    # Tạo ra các cột mới (VD: comfort_cozy_score)
    df[f'{mood}_score'] = df['all_text'].apply(lambda x: calculate_mood_score(x, keywords))

# 5. Chuẩn hóa dữ liệu (Normalization) - BƯỚC CỰC KỲ QUAN TRỌNG
# Biến các điểm số (có thể là 5, 10, 50) về thang điểm từ 0 đến 1 để đưa vào K-Means
mood_columns = [f'{mood}_score' for mood in mood_dictionaries.keys()]

for col in mood_columns:
    max_val = df[col].max()
    if max_val > 0:
        df[col] = df[col] / max_val # Chia cho giá trị lớn nhất để đưa về 0 -> 1

# 6. Lọc lại các cột cần thiết cho thuật toán phân cụm
# Chỉ giữ lại Tên quán, Tọa độ, Đánh giá, Mức giá và 5 cột Cảm xúc vừa tạo
columns_to_keep = ['title', 'location/lat', 'location/lng', 'totalScore'] + mood_columns

# Lấy những cột có tồn tại trong file (tránh báo lỗi nếu tên cột Apify hơi khác)
final_cols = [c for c in columns_to_keep if c in df.columns]
df_features = df[final_cols]

# 7. Lưu bộ dữ liệu vàng này lại
df_features.to_csv('data_cleaned/dataset_moodbite_features.csv', index=False, encoding='utf-8-sig')

print("Đã trích xuất đặc trưng xong!")
print("Xem thử 5 quán đầu tiên với điểm Mood Score:")
display(df_features.head())